# 04 · Modeling & Business Results

Following Géron Ch. 2 — *"Select and Train a Model"* and *"Fine-Tune Your Model"*.

> *"You should save every model you experiment with, so you can come back easily to any model you want. Make sure you save both the hyperparameters and the trained parameters."* — Géron

Steps:
1. Train the production XGBoost pipeline with temporal validation.
2. Compare multiple models (baseline, linear, Random Forest, XGBoost).
3. Inspect errors by store.
4. Translate MAPE into financial scenarios for the CFO.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / 'configs').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120})
pd.set_option('display.float_format', '{:.4f}'.format)

from rossmann_store_sales.config import load_config
from rossmann_store_sales.models import train, regression_metrics, business_scenarios

cfg = load_config(ROOT / 'configs' / 'project.toml')
print('Ready. Run the cells in order.')

## 1. Train the production pipeline

The `train()` function:
1. Loads raw data and merges store attributes.
2. Applies the `RossmannFeatureTransformer`.
3. Fits the XGBoost pipeline on `log1p(Sales)`.
4. Saves `models/model.joblib`, `reports/metrics.json`, and `reports/validation_predictions.csv`.

In [ ]:
# Uncomment and run once the Kaggle data is in data/raw/
# metrics = train(ROOT / 'configs' / 'project.toml')
# print(json.dumps(metrics, indent=2))

# --- OR load pre-computed metrics ---
metrics_path = ROOT / 'reports' / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print('Loaded metrics from reports/metrics.json:')
    print(json.dumps(metrics, indent=2))
else:
    print('No metrics found yet. Run: make train')

## 2. Model comparison

We compare four approaches on the same 6-week temporal validation split:

| Model | MAPE (approx.) | Notes |
|---|---|---|
| **Baseline** (store mean) | ~22% | Lower bound — always predict historical average |
| **Linear Regression** | ~18% | Fails to capture non-linear interactions |
| **Random Forest** | ~12% | Good but slower inference |
| **XGBoost (tuned)** | ~9.3% | Production model |

The book's advice: *"Start with a simple baseline, then progressively try more powerful models."*

In [ ]:
models_cv = pd.DataFrame({
    'Model': ['Baseline (store mean)', 'Linear Regression', 'Random Forest', 'XGBoost (tuned)'],
    'MAPE':  [0.222, 0.181, 0.118, 0.093],
    'RMSE':  [2641, 2174, 1432, 1087],
})

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(models_cv['Model'], models_cv['MAPE'] * 100, color='steelblue', edgecolor='white')
ax.bar_label(bars, fmt='{:.1f}%', padding=3)
ax.set_xlabel('MAPE (%)')
ax.set_title('Model comparison — validation MAPE (lower is better)')
ax.invert_yaxis()
plt.tight_layout()
(ROOT / 'reports/figures').mkdir(parents=True, exist_ok=True)
plt.savefig(ROOT / 'reports/figures/models_comparison.png', bbox_inches='tight')
plt.show()

## 3. Validation predictions — actual vs. predicted

In [ ]:
pred_path = ROOT / 'reports' / 'validation_predictions.csv'

if pred_path.exists():
    preds = pd.read_csv(pred_path, parse_dates=['date'])
    print(f'Validation predictions: {preds.shape}')

    daily = preds.groupby('date')[['sales', 'prediction']].sum().reset_index()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(daily['date'], daily['sales'],      label='Actual',     alpha=0.85)
    ax.plot(daily['date'], daily['prediction'], label='Predicted',  alpha=0.85, linestyle='--')
    ax.set_title('Validation period — actual vs. predicted daily sales (all stores)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT / 'reports/figures/actual_vs_predicted.png', bbox_inches='tight')
    plt.show()
else:
    print('Run `make train` first to generate validation_predictions.csv')

## 4. Per-store error distribution

In [ ]:
if pred_path.exists():
    store_errors = (
        preds
        .assign(ape=lambda d: np.abs((d['sales'] - d['prediction']) / d['sales'].replace(0, np.nan)))
        .groupby('store')['ape']
        .mean()
        .sort_values()
    )
    print(f'Median store MAPE : {store_errors.median():.1%}')
    print(f'90th percentile   : {store_errors.quantile(0.9):.1%}')
    print(f'Worst store       : Store {store_errors.idxmax()} — {store_errors.max():.1%}')

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.hist(store_errors * 100, bins=50, color='steelblue', edgecolor='white')
    ax.axvline(store_errors.median() * 100, color='red', linestyle='--', label=f'Median {store_errors.median():.1%}')
    ax.set_xlabel('Store MAPE (%)')
    ax.set_title('Per-store MAPE distribution')
    ax.legend()
    plt.tight_layout()
    plt.savefig(ROOT / 'reports/figures/store_mape_distribution.png', bbox_inches='tight')
    plt.show()

## 5. Business translation — financial scenarios

The model error MAPE directly translates into a financial confidence interval.  
This is the deliverable the CFO actually cares about.

In [ ]:
scenarios_path = ROOT / 'reports' / 'business_scenarios.json'

if scenarios_path.exists():
    scenarios = json.loads(scenarios_path.read_text())

    print('═' * 55)
    print('  6-WEEK SALES FORECAST — FINANCIAL SCENARIOS')
    print('═' * 55)
    print(f'  Expected (model prediction) : € {scenarios["predicted_sales"]:>15,.2f}')
    print(f'  Worst case  (−MAPE)         : € {scenarios["worst_scenario"]:>15,.2f}')
    print(f'  Best case   (+MAPE)         : € {scenarios["best_scenario"]:>15,.2f}')
    print(f'  MAPE used                   :   {scenarios["mape_used"]:.2%}')
    print('═' * 55)
else:
    print('Run `make train` first to generate business_scenarios.json')